In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# LexAI One-Click Databricks Runner

Run this notebook to initialize the notebook-06 engine and optionally expose API/UI from the same cluster session.

## What this notebook does
- Validates paths and cluster context
- Installs missing app dependencies (optional)
- Initializes notebook 06 runtime via adapter
- Runs smoke-test legal queries
- Optionally starts FastAPI and Streamlit

## Prerequisite
- `04_generate_embedding_Test.ipynb` and `06_High-precision_QA_Legal_Reasoning_Engine.ipynb` logic should be available in this repo.
- Embedding Delta data should exist in the configured Volume/table.


In [0]:
REPO_DIR_OVERRIDE = "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform"

In [0]:
# CELL 1: Runtime Flags (edit if needed)
AUTO_INSTALL_MISSING = True
RUN_SMOKE_TEST = True
START_FASTAPI = True
START_STREAMLIT = True

# Smoke test queries
SMOKE_TEST_QUERIES = [
    "Penalty for not wearing helmet in short within 120 words",
    "What does section 129 say in detail within 180 words",
]

# Server ports
FASTAPI_PORT = 8000
STREAMLIT_PORT = 8501

# Optional: override if repo location differs
REPO_DIR_OVERRIDE = ""

print("[CELL 1] Flags loaded")
print({
    "AUTO_INSTALL_MISSING": AUTO_INSTALL_MISSING,
    "RUN_SMOKE_TEST": RUN_SMOKE_TEST,
    "START_FASTAPI": START_FASTAPI,
    "START_STREAMLIT": START_STREAMLIT,
    "FASTAPI_PORT": FASTAPI_PORT,
    "STREAMLIT_PORT": STREAMLIT_PORT,
})


[CELL 1] Flags loaded
{'AUTO_INSTALL_MISSING': True, 'RUN_SMOKE_TEST': True, 'START_FASTAPI': True, 'START_STREAMLIT': True, 'FASTAPI_PORT': 8000, 'STREAMLIT_PORT': 8501}


In [0]:
# CELL 2: Resolve repo path and set working directory
import os
import sys
from pathlib import Path
from datetime import datetime


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def _repo_has_app_files(repo_dir: Path) -> bool:
    return (repo_dir / "apps" / "fastapi_app.py").exists() and (repo_dir / "apps" / "lexai06_notebook_adapter.py").exists()


def _safe_repo_scan(root: Path):
    skip_dirs = {"__pycache__", ".git", ".ipynb_checkpoints"}

    def _onerror(_err):
        return None

    for dirpath, dirnames, _filenames in os.walk(root, topdown=True, onerror=_onerror):
        dirnames[:] = [d for d in dirnames if d not in skip_dirs]
        repo_dir = Path(dirpath)
        try:
            if _repo_has_app_files(repo_dir):
                return repo_dir
        except Exception:
            continue
    return None


def _repo_from_notebook_context():
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        p = ctx.notebookPath().get()  # e.g. /Users/<email>/lexai-legal-rag-platform/notebooks/07_one_click_lexai_runner
        if not p:
            return None
        pp = Path(p)
        # expected: /Users/.../<repo>/notebooks/<nb>
        if len(pp.parts) >= 3:
            repo_guess = pp.parent.parent
            # filesystem view often under /Workspace
            fs_guess = Path("/Workspace") / Path(*repo_guess.parts[1:]) if str(repo_guess).startswith('/') else Path('/Workspace') / repo_guess
            return fs_guess
    except Exception:
        return None
    return None


def resolve_repo_dir() -> Path:
    # 1) explicit override
    if REPO_DIR_OVERRIDE and str(REPO_DIR_OVERRIDE).strip():
        p = Path(REPO_DIR_OVERRIDE.strip())
        if p.exists() and _repo_has_app_files(p):
            return p

    # 2) current working dir and parents
    cwd = Path(os.getcwd()).resolve()
    for candidate in [cwd] + list(cwd.parents):
        try:
            if _repo_has_app_files(candidate):
                return candidate
        except Exception:
            continue

    # 3) notebook context-derived guess
    ctx_guess = _repo_from_notebook_context()
    if ctx_guess is not None:
        try:
            if ctx_guess.exists() and _repo_has_app_files(ctx_guess):
                return ctx_guess
        except Exception:
            pass

    # 4) workspace scan
    for root in [Path('/Workspace/Repos'), Path('/Workspace/Users'), Path('/Workspace')]:
        if not root.exists():
            continue
        hit = _safe_repo_scan(root)
        if hit is not None:
            return hit

    raise FileNotFoundError(
        "Could not locate repo root containing apps/fastapi_app.py and apps/lexai06_notebook_adapter.py. "
        "Set REPO_DIR_OVERRIDE to your repo path (example: /Workspace/Users/<email>/lexai-legal-rag-platform)."
    )


REPO_DIR = resolve_repo_dir()
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

log(f"Repo root: {REPO_DIR}")
log(f"Working directory: {Path.cwd()}")
print("[CELL 2] OK")


[11:40:27] Repo root: /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform
[11:40:27] Working directory: /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform
[CELL 2] OK


In [0]:
# CELL 3: Optional dependency install (idempotent)
import importlib.metadata as ilm
import subprocess

REQ_FILE = Path("apps/requirements.txt")
if not REQ_FILE.exists():
    raise FileNotFoundError(f"Missing requirements file: {REQ_FILE}")

# App + notebook-06 runtime dependencies
required_pkgs = [
    "fastapi",
    "uvicorn",
    "streamlit",
    "requests",
    "pydantic",
    "sentence-transformers",
    "transformers",
    "accelerate",
    "mlflow",
    "databricks-sdk",
]
missing = []
for pkg in required_pkgs:
    try:
        ilm.version(pkg)
    except Exception:
        missing.append(pkg)

print("[CELL 3] Missing packages:", missing)
if missing and AUTO_INSTALL_MISSING:
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ_FILE)] + missing
    print("[CELL 3] Installing missing packages...")
    subprocess.check_call(cmd)
    print("[CELL 3] Installation completed")
elif missing and not AUTO_INSTALL_MISSING:
    raise RuntimeError(f"Missing required packages: {missing}. Set AUTO_INSTALL_MISSING=True.")
else:
    print("[CELL 3] All required packages already installed")


[CELL 3] Missing packages: []
[CELL 3] All required packages already installed


In [0]:
# CELL 4: Validate Spark and Databricks context
from pyspark.sql import SparkSession

spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
print("[CELL 4] Spark session ready:", bool(spark))

cluster_id = "unknown"
org_id = "unknown"
workspace_url = "unknown"

try:
    cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId")
except Exception:
    pass

try:
    org_id = spark.conf.get("spark.databricks.clusterUsageTags.orgId")
except Exception:
    pass

try:
    workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
except Exception:
    pass

print("[CELL 4] cluster_id:", cluster_id)
print("[CELL 4] org_id:", org_id)
print("[CELL 4] workspace_url:", workspace_url)


[CELL 4] Spark session ready: True
[CELL 4] cluster_id: 0301-061338-cb67io08-v2n
[CELL 4] org_id: unknown
[CELL 4] workspace_url: dbc-afb2e98d-d930.cloud.databricks.com


In [0]:
# CELL 5: Initialize notebook-06 engine through adapter (final)
from pathlib import Path
import importlib
import os
import apps.lexai06_notebook_adapter as _adapter

importlib.reload(_adapter)
NotebookEngine = _adapter.NotebookEngine

# Explicit priority list based on your provided workspace paths.
workspace_candidates = [
    "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine",
    "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine.ipynb",
    "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot",
    "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot.ipynb",
    str(Path(REPO_DIR) / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine.ipynb"),
    str(Path(REPO_DIR) / "apps" / "notebook_06_snapshot.ipynb"),
]

print("[CELL 5] Notebook candidates:")
for c in workspace_candidates:
    try:
        print(" -", c, "exists=", Path(c).exists())
    except Exception:
        print(" -", c, "exists=ERROR")

# Force adapter to try these paths first; adapter now supports workspace export fallback.
status = None
last_err = None
for cand in workspace_candidates:
    try:
        os.environ["LEXAI06_NOTEBOOK_PATH"] = cand
        engine = NotebookEngine(notebook_path=Path(cand))
        status = engine.initialize()
        print(f"[CELL 5] Initialized using candidate: {cand}")
        break
    except Exception as e:
        last_err = e
        print(f"[CELL 5] Candidate failed: {cand} -> {e}")

if status is None:
    raise RuntimeError(f"Engine initialization failed for all candidates. Last error: {last_err}")

print("[CELL 5] Engine initialized")
for k, v in status.items():
    print(f"  - {k}: {v}")

if not status.get("ready"):
    raise RuntimeError(f"Engine failed to initialize: {status}")


[CELL 5] Notebook candidates:
 - /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine exists= True
 - /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine.ipynb exists= False
 - /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot exists= True
 - /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot.ipynb exists= False
 - /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine.ipynb exists= False
 - /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot.ipynb exists= False
[CELL 5] Candidate failed: /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine -> Initialization failed at notebook cell unknown

---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
File <command-8746882894444235>, line 42
     39         print(f"[CELL 5] Candidate failed: {cand} -> {e}")
     41 if status is None:
---> 42     raise RuntimeError(f"Engine initialization failed for all candidates. Last error: {last_err}")
     44 print("[CELL 5] Engine initialized")
     45 for k, v in status.items():

RuntimeError: Engine initialization failed for all candidates. Last error: Initialization failed at notebook cell unknown: Notebook not found.
Searched candidates:
 - /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot.ipynb
 - /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine.ipynb
 - /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot.ipynb
Also searched roots for

In [0]:
# CELL 6: Smoke test queries (citation-rich output check)
if RUN_SMOKE_TEST:
    print("[CELL 6] Running smoke tests...")
    for idx, q in enumerate(SMOKE_TEST_QUERIES, start=1):
        print("=" * 90)
        print(f"[{idx}] Query: {q}")
        out = engine.answer_query(q)
        print("Mode:", out.get("mode"))
        print("Source:", out.get("source"))
        print("Confidence:", out.get("confidence"))
        print("Sections:", out.get("sections", []))
        print("Citations:", out.get("citations", [])[:5])
        print("Latency:", out.get("latency_ms", {}))
        print("Answer:")
        print(out.get("answer", ""))
    print("[CELL 6] Smoke tests done")
else:
    print("[CELL 6] RUN_SMOKE_TEST=False -> skipped")


In [0]:
# CELL 7: Start FastAPI server (background thread)
import threading
import uvicorn

FASTAPI_SERVER = globals().get("FASTAPI_SERVER")
FASTAPI_THREAD = globals().get("FASTAPI_THREAD")

if START_FASTAPI:
    if FASTAPI_THREAD is not None and FASTAPI_THREAD.is_alive():
        print(f"[CELL 7] FastAPI already running on port {FASTAPI_PORT}")
    else:
        from apps.fastapi_app import app

        config = uvicorn.Config(app, host="0.0.0.0", port=int(FASTAPI_PORT), log_level="info")
        FASTAPI_SERVER = uvicorn.Server(config)
        FASTAPI_THREAD = threading.Thread(target=FASTAPI_SERVER.run, daemon=True)
        FASTAPI_THREAD.start()
        globals()["FASTAPI_SERVER"] = FASTAPI_SERVER
        globals()["FASTAPI_THREAD"] = FASTAPI_THREAD
        print(f"[CELL 7] FastAPI started on 0.0.0.0:{FASTAPI_PORT}")

    print("[CELL 7] Local health URL:", f"http://127.0.0.1:{FASTAPI_PORT}/health")

    try:
        if workspace_url != "unknown" and org_id != "unknown" and cluster_id != "unknown":
            proxy_url = f"https://{workspace_url}/driver-proxy/o/{org_id}/{cluster_id}/{FASTAPI_PORT}/health"
            print("[CELL 7] Driver proxy URL:", proxy_url)
    except Exception:
        pass
else:
    print("[CELL 7] START_FASTAPI=False -> skipped")


In [0]:
# CELL 8: Optional FastAPI smoke call
import requests

if START_FASTAPI:
    try:
        h = requests.get(f"http://127.0.0.1:{FASTAPI_PORT}/health", timeout=30)
        print("[CELL 8] /health status:", h.status_code)
        print(h.json())

        payload = {
            "query": "What is the penalty for not wearing a helmet?",
            "style": "short",
            "word_limit": 120,
        }
        r = requests.post(f"http://127.0.0.1:{FASTAPI_PORT}/v1/legal/answer", json=payload, timeout=180)
        print("[CELL 8] /v1/legal/answer status:", r.status_code)
        print("[CELL 8] answer preview:", r.json().get("answer", "")[:500])
    except Exception as e:
        print("[CELL 8] API call failed:", e)
else:
    print("[CELL 8] START_FASTAPI=False -> skipped")


In [0]:
# CELL 9: Optional Streamlit start (blocking cell)
import os
import subprocess

if START_STREAMLIT:
    os.environ["LEXAI_API_BASE_URL"] = f"http://127.0.0.1:{FASTAPI_PORT}"
    cmd = [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "apps/streamlit_app.py",
        "--server.port",
        str(STREAMLIT_PORT),
        "--server.address",
        "0.0.0.0",
    ]
    print("[CELL 9] Starting Streamlit:", " ".join(cmd))
    if workspace_url != "unknown" and org_id != "unknown" and cluster_id != "unknown":
        ui_url = f"https://{workspace_url}/driver-proxy/o/{org_id}/{cluster_id}/{STREAMLIT_PORT}/"
        print("[CELL 9] Streamlit URL:", ui_url)
    print("[CELL 9] This cell is blocking while Streamlit is running.")
    subprocess.call(cmd)
else:
    print("[CELL 9] START_STREAMLIT=False -> skipped")


In [0]:
# CELL 10: Stop helper (run when needed)
if "FASTAPI_SERVER" in globals() and globals().get("FASTAPI_SERVER") is not None:
    globals()["FASTAPI_SERVER"].should_exit = True
    print("[CELL 10] FastAPI stop requested")
else:
    print("[CELL 10] FastAPI was not running")
